In [1]:
import pandas as pd
import numpy as np
import psycopg

In [2]:
conn = psycopg.connect("dbname=dailyedge_development")

print("Connected to dailyedge_development")

Connected to dailyedge_development


In [3]:
query = """
SELECT
    MIN(timestamp) AS first_timestamp,
    MAX(timestamp) AS last_timestamp,
    COUNT(*) AS total_rows
FROM CANDLES;
"""

db_info = pd.read_sql(query, conn)
db_info

/tmp/ipykernel_27024/1321250083.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_info = pd.read_sql(query, conn)


,first_timestamp,last_timestamp,total_rows
0,2008-12-11 01:38:00,2026-07-08,5891410


In [4]:
query = """
SELECT
    timestamp,
    open,
    high,
    low,
    close,
    volume
FROM CANDLES
WHERE timestamp >= '2025-09-01 08:30:00'
  AND timestamp <= '2026-07-07 15:15:00'
  AND timestamp::time BETWEEN '08:30:00' AND '15:15:00'
ORDER BY timestamp;
"""

candles = pd.read_sql(query, conn)

print("Rows:", len(candles))
print("First timestamp:", candles["timestamp"].min())
print("Last timestamp:", candles["timestamp"].max())

/tmp/ipykernel_27024/464963839.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  candles = pd.read_sql(query, conn)


Rows: 87179
First timestamp: 2025-09-01 08:30:00
Last timestamp: 2026-07-07 15:15:00


In [5]:
candles["Date"] = candles["timestamp"].dt.date
candles["Day"] = candles["timestamp"].dt.day_name()

session_counts = (
    candles.groupby(["Date", "Day"])
    .size()
    .reset_index(name="Candles")
)

print("Sessions:", len(session_counts))
print()
print(session_counts["Day"].value_counts().sort_index())

Sessions: 219

Day
Friday       43
Monday       45
Thursday     42
Tuesday      45
Wednesday    44
Name: count, dtype: int64


In [6]:
def evaluate_raw_cleanliness(session, stop=35, target=70):
    session = session.sort_values("timestamp").reset_index(drop=True)

    opening_price = session.iloc[0]["open"]

    upper_target = opening_price + target
    lower_target = opening_price - target

    upper_hits = session.index[session["high"] >= upper_target]
    lower_hits = session.index[session["low"] <= lower_target]

    if len(upper_hits) == 0 and len(lower_hits) == 0:
        return "Neither", None

    if len(upper_hits) == 0:
        direction = "Short"
        target_idx = lower_hits[0]
    elif len(lower_hits) == 0:
        direction = "Long"
        target_idx = upper_hits[0]
    else:
        first_upper = upper_hits[0]
        first_lower = lower_hits[0]

        if first_upper == first_lower:
            return "Unknown", "Unknown"

        if first_upper < first_lower:
            direction = "Long"
            target_idx = first_upper
        else:
            direction = "Short"
            target_idx = first_lower

    before_target = session.loc[:target_idx]

    if direction == "Long":
        adverse_level = opening_price - stop

        adverse_hits = before_target.index[
            before_target["low"] <= adverse_level
        ]

    else:
        adverse_level = opening_price + stop

        adverse_hits = before_target.index[
            before_target["high"] >= adverse_level
        ]

    if len(adverse_hits) == 0:
        cleanliness = "Clean"
    else:
        first_adverse = adverse_hits[0]

        if first_adverse == target_idx:
            cleanliness = "Unknown"
        else:
            cleanliness = "Not Clean"

    return direction, cleanliness

In [11]:
stop_target_pairs = [
    (15, 25),
    (25, 50),
    (35, 70),
    (50, 70),
    (50, 100),
    (75, 100),
    (75, 150),
]

all_raw_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        direction, cleanliness = evaluate_raw_cleanliness(
            session,
            stop=stop,
            target=target
        )

        all_raw_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Direction": direction,
            "Cleanliness": cleanliness
        })

all_raw_results = pd.DataFrame(all_raw_results)

print("Rows:", len(all_raw_results))
print()
print(
    all_raw_results
    .groupby(["Stop", "Target"])["Cleanliness"]
    .value_counts(dropna=False)
)

Rows: 1533

Stop  Target  Cleanliness
15    25      Clean          139
              Unknown         53
              Not Clean       26
              NaN              1
25    50      Clean          149
              Not Clean       60
              Unknown          7
              NaN              3
35    70      Clean          144
              Not Clean       69
              NaN              6
50    70      Clean          174
              Not Clean       38
              NaN              6
              Unknown          1
      100     Clean          140
              Not Clean       68
              NaN             11
75    100     Clean          181
              Not Clean       27
              NaN             11
      150     Clean          129
              Not Clean       50
              NaN             40
Name: count, dtype: int64


In [14]:
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday"
]

resolved_raw = all_raw_results[
    all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
].copy()

resolved_raw["Clean"] = resolved_raw["Cleanliness"] == "Clean"

raw_weekday_summary = (
    resolved_raw
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Clean", "size"),
        Clean=("Clean", "sum")
    )
    .reset_index()
)

raw_weekday_summary["Clean Rate"] = (
    raw_weekday_summary["Clean"]
    / raw_weekday_summary["Resolved"]
    * 100
)

raw_cleanliness_table = (
    raw_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Clean Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved = (
    all_raw_results[
        ~all_raw_results["Cleanliness"].isin(["Clean", "Not Clean"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

raw_cleanliness_table["Unresolved"] = unresolved

raw_cleanliness_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       90.62    83.33      83.78     79.31   83.87          54
25   50       69.05    77.27      77.27     74.36   57.50          10
35   70       72.09    77.78      68.18     65.85   52.50           6
50   70       86.05    84.44      79.55     80.49   79.49           7
     100      73.81    63.64      60.47     72.50   66.67          11
75   100      92.86    77.27      90.70     90.00   84.62          11
     150      81.82    55.88      80.49     67.57   73.53          40

In [15]:
def evaluate_continuous_trail(
    rth,
    opening_price,
    target_distance,
    trail_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        highest_price = opening_price
        trailing_stop = opening_price - trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price <= old_stop:
                return "Failure"

            new_highest = max(highest_price, high)
            new_stop = new_highest - trail_distance

            old_stop_hit = low <= old_stop
            new_stop_hit = low <= new_stop
            target_hit = high >= target

            # Low first
            if old_stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif new_stop_hit:
                high_first = "Failure"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if {low_first, high_first} == {"Continue", "Failure"}:
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if low_first == "Continue" and high_first == "Continue":
                highest_price = new_highest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        lowest_price = opening_price
        trailing_stop = opening_price + trail_distance

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            old_stop = trailing_stop

            # If the candle opens through the existing stop,
            # the trade is already stopped before anything else happens.
            if open_price >= old_stop:
                return "Failure"

            new_lowest = min(lowest_price, low)
            new_stop = new_lowest + trail_distance

            old_stop_hit = high >= old_stop
            new_stop_hit = high >= new_stop
            target_hit = low <= target

            # High first
            if old_stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif new_stop_hit:
                low_first = "Failure"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if {high_first, low_first} == {"Continue", "Failure"}:
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if high_first == "Continue" and low_first == "Continue":
                lowest_price = new_lowest
                trailing_stop = new_stop
                continue

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [16]:
def get_first_target(rth, opening_price, target_distance):
    upper_target = opening_price + target_distance
    lower_target = opening_price - target_distance

    target_hits = rth[
        (rth["high"] >= upper_target) |
        (rth["low"] <= lower_target)
    ]

    if target_hits.empty:
        return "Neither", None

    first_hit = target_hits.iloc[0]

    hit_upper = first_hit["high"] >= upper_target
    hit_lower = first_hit["low"] <= lower_target

    if hit_upper and hit_lower:
        return "Ambiguous", first_hit["timestamp"]

    result = "Long" if hit_upper else "Short"

    return result, first_hit["timestamp"]

In [18]:
all_trail_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_continuous_trail(
            session,
            opening_price,
            target_distance=target,
            trail_distance=stop
        )

        all_trail_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_trail_results = pd.DataFrame(all_trail_results)

print("Rows:", len(all_trail_results))
print()
print(
    all_trail_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1533

Stop  Target  Outcome  
15    25      Success      127
              Unknown       39
              Failure       36
              Ambiguous     16
              Neither        1
25    50      Failure      108
              Success       97
              Unknown        9
              Neither        3
              Ambiguous      2
35    70      Failure      131
              Success       80
              Neither        6
              Unknown        2
50    70      Success      120
              Failure       92
              Neither        6
              Unknown        1
      100     Failure      145
              Success       61
              Neither       11
              Unknown        2
75    100     Success      113
              Failure       93
              Neither       11
              Unknown        2
      150     Failure      118
              Success       61
              Neither       40
Name: count, dtype: int64


In [19]:
resolved_trail = all_trail_results[
    all_trail_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_trail["Success"] = resolved_trail["Outcome"] == "Success"

trail_weekday_summary = (
    resolved_trail
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

trail_weekday_summary["Survival Rate"] = (
    trail_weekday_summary["Success"]
    / trail_weekday_summary["Resolved"]
    * 100
)

trail_survival_table = (
    trail_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_trail = (
    all_trail_results[
        ~all_trail_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

trail_survival_table["Unresolved"] = unresolved_trail

trail_survival_table.round(2)

Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       83.87    75.00      77.78     75.86   77.42          56
25   50       38.10    51.16      53.49     51.35   42.50          14
35   70       39.53    40.91      37.21     36.59   35.00           8
50   70       62.79    60.00      54.55     58.54   46.15           7
     100      41.46    25.58      23.26     30.00   28.21          13
75   100      76.19    51.16      48.84     53.85   43.59          13
     150      42.42    32.35      26.83     43.24   26.47          40

In [20]:
def evaluate_one_move_breakeven(
    rth,
    opening_price,
    target_distance,
    stop_distance
):
    result, target_time = get_first_target(
        rth,
        opening_price,
        target_distance
    )

    if result in ["Neither", "Ambiguous"]:
        return result

    # -------------------------
    # LONG
    # -------------------------

    if result == "Long":
        target = opening_price + target_distance
        threshold = opening_price + stop_distance

        initial_stop = opening_price - stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price <= stop:
                return "Failure"

            target_hit = high >= target
            stop_hit = low <= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = high >= threshold

            # Low first
            if stop_hit:
                low_first = "Failure"
            elif target_hit:
                low_first = "Success"
            else:
                low_first = "Continue"

            # High first
            if target_hit:
                high_first = "Success"
            elif threshold_hit:
                if low <= opening_price:
                    high_first = "Failure"
                else:
                    high_first = "Continue"
            else:
                high_first = "Continue"

            if low_first == high_first:
                if low_first == "Success":
                    return "Success"
                if low_first == "Failure":
                    return "Failure"

            if {low_first, high_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    # -------------------------
    # SHORT
    # -------------------------

    if result == "Short":
        target = opening_price - target_distance
        threshold = opening_price - stop_distance

        initial_stop = opening_price + stop_distance
        stop = initial_stop

        moved_to_breakeven = False

        for _, candle in rth.iterrows():
            open_price = candle["open"]
            high = candle["high"]
            low = candle["low"]

            if open_price >= stop:
                return "Failure"

            target_hit = low <= target
            stop_hit = high >= stop

            if moved_to_breakeven:
                if target_hit and stop_hit:
                    return "Unknown"
                if target_hit:
                    return "Success"
                if stop_hit:
                    return "Failure"
                continue

            threshold_hit = low <= threshold

            # High first
            if stop_hit:
                high_first = "Failure"
            elif target_hit:
                high_first = "Success"
            else:
                high_first = "Continue"

            # Low first
            if target_hit:
                low_first = "Success"
            elif threshold_hit:
                if high >= opening_price:
                    low_first = "Failure"
                else:
                    low_first = "Continue"
            else:
                low_first = "Continue"

            if high_first == low_first:
                if high_first == "Success":
                    return "Success"
                if high_first == "Failure":
                    return "Failure"

            if {high_first, low_first} == {"Success", "Failure"}:
                return "Unknown"

            if threshold_hit:
                moved_to_breakeven = True
                stop = opening_price

            if candle["timestamp"] >= target_time:
                return "Unknown"

    return None

In [22]:
all_be_results = []

for stop, target in stop_target_pairs:
    for date, session in candles.groupby("Date"):
        session = session.sort_values("timestamp").reset_index(drop=True)
        opening_price = session.iloc[0]["open"]

        outcome = evaluate_one_move_breakeven(
            session,
            opening_price,
            target_distance=target,
            stop_distance=stop
        )

        all_be_results.append({
            "Date": date,
            "Day": session.iloc[0]["Day"],
            "Stop": stop,
            "Target": target,
            "Outcome": outcome
        })

all_be_results = pd.DataFrame(all_be_results)

print("Rows:", len(all_be_results))
print()
print(
    all_be_results
    .groupby(["Stop", "Target"])["Outcome"]
    .value_counts(dropna=False)
)

Rows: 1533

Stop  Target  Outcome  
15    25      Success      139
              Unknown       38
              Failure       25
              Ambiguous     16
              Neither        1
25    50      Success      141
              Failure       68
              Unknown        5
              Neither        3
              Ambiguous      2
35    70      Success      145
              Failure       67
              Neither        6
              Unknown        1
50    70      Success      182
              Failure       30
              Neither        6
              Unknown        1
      100     Success      128
              Failure       79
              Neither       11
              Unknown        1
75    100     Success      172
              Failure       35
              Neither       11
              Unknown        1
      150     Success      112
              Failure       67
              Neither       40
Name: count, dtype: int64


In [23]:
resolved_be = all_be_results[
    all_be_results["Outcome"].isin(["Success", "Failure"])
].copy()

resolved_be["Success"] = resolved_be["Outcome"] == "Success"

be_weekday_summary = (
    resolved_be
    .groupby(["Stop", "Target", "Day"])
    .agg(
        Resolved=("Success", "size"),
        Success=("Success", "sum")
    )
    .reset_index()
)

be_weekday_summary["Survival Rate"] = (
    be_weekday_summary["Success"]
    / be_weekday_summary["Resolved"]
    * 100
)

be_survival_table = (
    be_weekday_summary
    .pivot(
        index=["Stop", "Target"],
        columns="Day",
        values="Survival Rate"
    )
    .reindex(columns=weekday_order)
)

unresolved_be = (
    all_be_results[
        ~all_be_results["Outcome"].isin(["Success", "Failure"])
    ]
    .groupby(["Stop", "Target"])
    .size()
)

be_survival_table["Unresolved"] = unresolved_be

be_survival_table.round(2)


Day          Monday  Tuesday  Wednesday  Thursday  Friday  Unresolved
Stop Target                                                          
15   25       87.50    82.86      81.08     82.76   90.32          55
25   50       66.67    68.18      70.45     74.36   57.50          10
35   70       72.09    68.89      68.18     73.17   58.97           7
50   70       90.70    88.89      86.36     90.24   71.79           7
     100      78.57    60.47      58.14     65.00   46.15          12
75   100      88.10    86.05      79.07     80.00   82.05          12
     150      60.61    61.76      58.54     64.86   67.65          40